# Практическое занятие 9
## Методы оценки плотности для выявления аномалий в кибербезопасности

Продолжительность: 80 минут. Метки `scenario` и `is_attack` используются только для учебной проверки качества и не передаются моделям как входные признаки.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
%config InlineBackend.figure_format = 'retina'

In [ ]:
FEATURES = [
    'connections_per_min',
    'unique_destinations',
    'outbound_mb'
]

data_path = Path('cyber_density_anomalies.csv')
df = pd.read_csv(data_path)

reference_df = df.loc[df['dataset_part'] == 'reference_normal'].copy()
monitoring_df = df.loc[df['dataset_part'] == 'monitoring'].copy()
print('Вся выборка:', df.shape)
print('Профиль нормы:', reference_df.shape)
print('Мониторинг:', monitoring_df.shape)

def print_detection_result(method_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(method_name)
    print(f'TP — обнаруженные атаки: {tp}')
    print(f'FP — ложные тревоги: {fp}')
    print(f'FN — пропущенные атаки: {fn}')
    print(f'TN — корректно распознанная норма: {tn}')

def fit_hbos(X, bins=12, alpha=1.0):
    models = []
    n_objects = len(X)
    for feature_index in range(X.shape[1]):
        counts, edges = np.histogram(X[:, feature_index], bins=bins)
        probabilities = (counts + alpha) / (n_objects + alpha * bins)
        models.append({'edges': edges, 'probabilities': probabilities})
    return models

def hbos_score_samples(models, X, epsilon=1e-12):
    scores = np.zeros(len(X))
    for feature_index, model in enumerate(models):
        edges = model['edges']
        probabilities = model['probabilities']
        bin_indices = np.searchsorted(edges, X[:, feature_index], side='right') - 1
        bin_indices[X[:, feature_index] == edges[-1]] = len(probabilities) - 1
        inside = (bin_indices >= 0) & (bin_indices < len(probabilities))
        point_probabilities = np.full(len(X), epsilon)
        point_probabilities[inside] = probabilities[bin_indices[inside]]
        scores -= np.log(point_probabilities)
    return scores

def scenario_table(prediction):
    result = monitoring_df.loc[monitoring_df['is_attack'] == 1, ['scenario']].copy()
    result['detected'] = np.asarray(prediction)[monitoring_df['is_attack'].to_numpy() == 1]
    return result.groupby('scenario')['detected'].agg(['sum', 'count'])

## Задание 1. Подготовка данных и режимов работы — 3 балла
1. Выполните аудит структуры, пропусков и дубликатов.
2. Подтвердите состав `reference_normal` и `monitoring`.
3. Подготовьте стандартизованную смешанную выборку.
4. Разделите подтверждённую норму на 75% fit и 25% calibration со стратификацией по `traffic_type`.
5. Объясните, почему scaler в режиме профиля нормы обучается только на `normal_fit_df`.

In [ ]:
# аудит и подготовка X_unsupervised, X_normal_fit,
# X_normal_calibration и X_monitoring.
# Подсказка: train_test_split(..., stratify=reference_df['traffic_type']).

## Задание 2. KDE в двух режимах — 5 баллов
Используйте гауссовское ядро и `bandwidth=0.35`. Рассчитайте score как отрицательный логарифм плотности. Для смешанной выборки примените квантиль `1 − 0.055`; для профиля нормы — 99-й процентиль калибровочных нормальных оценок. Выведите пороги, TP, FP, FN, TN и обнаружение по сценариям.

In [ ]:
# реализуйте KDE в двух режимах.
# score = -model.score_samples(X)
# Метка аномалии: score > threshold

## Задание 3. GMM в двух режимах — 4 балла
Используйте три компоненты, полные ковариационные матрицы, `reg_covar=1e-6`, `random_state=42`, `n_init=1`. Рассчитайте `−log p(x)`, выберите пороги по тем же правилам и сопоставьте два режима.

In [ ]:
# реализуйте GMM в двух режимах.
# Используйте GaussianMixture и score_samples().

## Задание 4. HBOS в двух режимах — 4 балла
Используйте ручные функции лекции, 12 интервалов и сглаживание `alpha=1.0`. В смешанном режиме задайте долю 0.05; в режиме профиля нормы примените 99-й процентиль калибровочных оценок. Установите, какие атаки пропущены, и объясните результат через предположение независимости признаков.

In [ ]:
# примените fit_hbos() и hbos_score_samples()
# к смешанной выборке и к профилю нормы.

## Задание 5. Сравнение методов и вывод SOC — 4 балла
Сведите TP, FP, FN, TN для шести комбинаций метода и режима в одну таблицу, постройте показанные в лекции гистограммы KDE-оценок с порогами и подготовьте аналитическую записку SOC из 5–7 предложений. Укажите набор данных, методы, пороги, результаты, ограничения и рекомендуемую проверку.

In [ ]:
# сформируйте таблицу сравнения и постройте
# гистограммы KDE-оценок с порогами для двух режимов.

### Аналитическая записка SOC
За период / на наборе данных …

Применены методы … с порогами …

Выявлено …

Наиболее значимые наблюдения …

Ограничения …

Рекомендуется …